In [2]:
import os
import random

import jax
import jax.numpy as jnp

os.environ['TORCH_USE_CUDA_DSA'] = "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_FLAGS"] = "--xla_gpu_force_compilation_parallelism=1"

# Verify with the new 2026 JAX backend check
import jax
from jax.extend import backend
try:
    print(f"Active Backend: {backend.get_backend().platform}")
    print(f"Devices: {jax.devices()}")
except Exception as e:
    print(f"Error: {e}")

import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data

import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torchvision import datasets, transforms
import torchvision.transforms as transforms

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import pennylane as qml

from tqdm import tqdm
from matplotlib import cm
from scipy.linalg import sqrtm
from sklearn.decomposition import PCA

import tensorcircuit as tc

K = tc.set_backend("jax")
print("K:", K)

import time

Active Backend: gpu
Devices: [CudaDevice(id=0)]
K: jax_backend


In [3]:
class ParserLocal:
    def __init__(self):
        self.props = {}

    def add_argument(self, key, value=None, required=None, type=None, default=None, action=None, help=None):
        self.props[key[2:]] = value if value is not None else default

    def get_props(self):
        return self.props

In [4]:
class dotdict(dict):
    """dot.notation access to dictionary attributes"""
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

In [5]:
parser = ParserLocal()
parser.add_argument('--dataset', value='mnist', required=True, help='cifar10 | lsun | mnist |imagenet | folder | lfw | fake')
parser.add_argument('--dataroot', value='', required=False, help='path to dataset')
parser.add_argument('--workers', type=int, help='number of data loading workers', default=2)
parser.add_argument('--batchSize', type=int, default=64, help='input batch size')
parser.add_argument('--imageSize', type=int, default=64, help='the height / width of the input image to network')
parser.add_argument('--nz', type=int, default=100, help='size of the latent z vector')
parser.add_argument('--ngf', type=int, default=64, help='number of generator filters')
parser.add_argument('--ndf', type=int, default=64, help='number of discriminator filters')
parser.add_argument('--niter', type=int, default=25, help='number of epochs to train for')
parser.add_argument('--lr', type=float, default=0.0002, help='learning rate, default=0.0002')
parser.add_argument('--beta1', type=float, default=0.5, help='beta1 for adam. default=0.5')
parser.add_argument('--dry-run', action='store_true', help='check a single training cycle works')
parser.add_argument('--ngpu', type=int, default=1, help='number of GPUs to use')
parser.add_argument('--netG', default='', help="path to netG (to continue training)")
parser.add_argument('--netD', default='', help="path to netD (to continue training)")
parser.add_argument('--outf', default='./classical_GAN', help='folder to output images and model checkpoints')
parser.add_argument('--manualSeed', value=42, type=int, help='manual seed')
parser.add_argument('--classes', default='bedroom', help='comma separated list of classes for the lsun data set')
parser.add_argument('--accel', action='store_true', default=True, help='enables accelerator')

In [6]:
opt = dotdict(parser.get_props())
print(opt)

{'dataset': 'mnist', 'dataroot': '', 'workers': 2, 'batchSize': 64, 'imageSize': 64, 'nz': 100, 'ngf': 64, 'ndf': 64, 'niter': 25, 'lr': 0.0002, 'beta1': 0.5, 'dry-run': None, 'ngpu': 1, 'netG': '', 'netD': '', 'outf': './classical_GAN', 'manualSeed': 42, 'classes': 'bedroom', 'accel': True}


In [7]:
try:
    os.makedirs(opt.outf)
except OSError:
    pass

In [8]:
if opt.manualSeed is None:
    opt.manualSeed = random.randint(1, 10000)
print("Random Seed: ", opt.manualSeed)
random.seed(opt.manualSeed)
torch.manual_seed(opt.manualSeed)

Random Seed:  42


In [9]:
cudnn.benchmark = True

if opt.accel and torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [10]:
if opt.dataroot is None and str(opt.dataset).lower() != 'fake':
    raise ValueError("`dataroot` parameter is required for dataset \"%s\"" % opt.dataset)

if opt.dataset in ['imagenet', 'folder', 'lfw']:
    # folder dataset
    dataset = dset.ImageFolder(root=opt.dataroot,
                               transform=transforms.Compose([
                                   transforms.Resize(opt.imageSize),
                                   transforms.CenterCrop(opt.imageSize),
                                   transforms.ToTensor(),
                                   transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                               ]))
    nc=3
elif opt.dataset == 'lsun':
    classes = [ c + '_train' for c in opt.classes.split(',')]
    dataset = dset.LSUN(root=opt.dataroot, classes=classes,
                        transform=transforms.Compose([
                            transforms.Resize(opt.imageSize),
                            transforms.CenterCrop(opt.imageSize),
                            transforms.ToTensor(),
                            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                        ]))
    nc=3
elif opt.dataset == 'cifar10':
    dataset = dset.CIFAR10(root=opt.dataroot, download=True,
                           transform=transforms.Compose([
                               transforms.Resize(opt.imageSize),
                               transforms.ToTensor(),
                               transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                           ]))
    nc=3

elif opt.dataset == 'mnist':
        dataset = dset.MNIST(root=opt.dataroot, download=True,
                           transform=transforms.Compose([
                               transforms.Resize(opt.imageSize),
                               transforms.ToTensor(),
                               transforms.Normalize((0.5,), (0.5,)),
                           ]))
        nc=1

elif opt.dataset == 'fake':
    dataset = dset.FakeData(image_size=(3, opt.imageSize, opt.imageSize),
                            transform=transforms.ToTensor())
    nc=3

assert dataset
dataloader = torch.utils.data.DataLoader(dataset, batch_size=opt.batchSize,
                                         shuffle=True, num_workers=int(opt.workers))

In [11]:
ngpu = int(opt.ngpu)
nz = int(opt.nz)
ngf = int(opt.ngf)
ndf = int(opt.ndf)

In [12]:
# custom weights initialization called on netG and netD
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        torch.nn.init.normal_(m.weight, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        torch.nn.init.normal_(m.weight, 1.0, 0.02)
        torch.nn.init.zeros_(m.bias)

In [12]:
class Generator(nn.Module):
    def __init__(self, ngpu):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(     nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2,     ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 32 x 32
            nn.ConvTranspose2d(    ngf,      nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # state size. (nc) x 64 x 64
        )

    def forward(self, input):
        
        if (input.is_cuda or input.is_xpu) and self.ngpu > 1:
            output = nn.parallel.data_parallel(self.main, input, range(self.ngpu))
        else:
            output = self.main(input)
        return output

In [13]:
class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is (nc) x 64 x 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf) x 32 x 32
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*2) x 16 x 16
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*4) x 8 x 8
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*8) x 4 x 4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        if (input.is_cuda or input.is_xpu) and self.ngpu > 1:
            output = nn.parallel.data_parallel(self.main, input, range(self.ngpu))
        else:
            output = self.main(input)

        return output.view(-1, 1).squeeze(1)

In [14]:
netG = Generator(ngpu).to(device)
netG.apply(weights_init)
if opt.netG != '':
    netG.load_state_dict(torch.load(opt.netG))
print(netG)

Generator(
  (main): Sequential(
    (0): ConvTranspose2d(100, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): ConvTranspose2d(64, 1, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (13): Tanh()
  )
)


In [15]:
netD = Discriminator(ngpu).to(device)
netD.apply(weights_init)
if opt.netD != '':
    netD.load_state_dict(torch.load(opt.netD))
print(netD)

Discriminator(
  (main): Sequential(
    (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (9): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Conv2d(512, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (12): Sigmoid()
  )
)


In [16]:
criterion = nn.BCELoss()

fixed_noise = torch.randn(opt.batchSize, nz, 1, 1, device=device)
real_label = 1
fake_label = 0

In [17]:
optimizerD = optim.Adam(netD.parameters(), lr=opt.lr, betas=(opt.beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=opt.lr, betas=(opt.beta1, 0.999))

In [18]:
if opt.dry_run:
    opt.niter = 1

## train

In [56]:
for epoch in range(opt.niter):
    for i, data in enumerate(dataloader, 0):
        ############################
        # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
        ###########################
        # train with real
        netD.zero_grad()
        real_cpu = data[0].to(device)
        batch_size = real_cpu.size(0)
        label = torch.full((batch_size,), real_label,
                           dtype=real_cpu.dtype, device=device)

        output = netD(real_cpu)
        errD_real = criterion(output, label)
        errD_real.backward()
        D_x = output.mean().item()

        # train with fake
        noise = torch.randn(batch_size, nz, 1, 1, device=device)
        fake = netG(noise)
        label.fill_(fake_label)
        output = netD(fake.detach())
        errD_fake = criterion(output, label)
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        errD = errD_real + errD_fake
        optimizerD.step()

        ############################
        # (2) Update G network: maximize log(D(G(z)))
        ###########################
        netG.zero_grad()
        label.fill_(real_label)  # fake labels are real for generator cost
        output = netD(fake)
        errG = criterion(output, label)
        errG.backward()
        D_G_z2 = output.mean().item()
        optimizerG.step()

        print('[%d/%d][%d/%d] Loss_D: %.4f Loss_G: %.4f D(x): %.4f D(G(z)): %.4f / %.4f'
              % (epoch, opt.niter, i, len(dataloader),
                 errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))
        if i % 100 == 0:
            vutils.save_image(real_cpu,
                    '%s/real_samples.png' % opt.outf,
                    normalize=True)
            fake = netG(fixed_noise)
            vutils.save_image(fake.detach(),
                    '%s/fake_samples_epoch_%03d.png' % (opt.outf, epoch),
                    normalize=True)

        if opt.dry_run:
            break
    # do checkpointing
    torch.save(netG.state_dict(), '%s/netG_epoch_%d.pth' % (opt.outf, epoch))
    torch.save(netD.state_dict(), '%s/netD_epoch_%d.pth' % (opt.outf, epoch))

[0/25][0/938] Loss_D: 2.0064 Loss_G: 4.3990 D(x): 0.4342 D(G(z)): 0.5784 / 0.0196
[0/25][1/938] Loss_D: 1.3938 Loss_G: 6.1450 D(x): 0.9985 D(G(z)): 0.6875 / 0.0034
[0/25][2/938] Loss_D: 0.6545 Loss_G: 7.0559 D(x): 0.9960 D(G(z)): 0.4121 / 0.0014
[0/25][3/938] Loss_D: 0.2373 Loss_G: 7.0132 D(x): 0.9683 D(G(z)): 0.1532 / 0.0013
[0/25][4/938] Loss_D: 0.1930 Loss_G: 6.8652 D(x): 0.9596 D(G(z)): 0.1236 / 0.0017
[0/25][5/938] Loss_D: 0.2224 Loss_G: 7.2944 D(x): 0.9540 D(G(z)): 0.1431 / 0.0011
[0/25][6/938] Loss_D: 0.1572 Loss_G: 7.8656 D(x): 0.9722 D(G(z)): 0.1121 / 0.0007
[0/25][7/938] Loss_D: 0.1582 Loss_G: 7.8274 D(x): 0.9591 D(G(z)): 0.1013 / 0.0007
[0/25][8/938] Loss_D: 0.1596 Loss_G: 8.4081 D(x): 0.9691 D(G(z)): 0.1107 / 0.0003
[0/25][9/938] Loss_D: 0.0621 Loss_G: 7.8893 D(x): 0.9826 D(G(z)): 0.0404 / 0.0006
[0/25][10/938] Loss_D: 0.1159 Loss_G: 8.0872 D(x): 0.9747 D(G(z)): 0.0807 / 0.0004
[0/25][11/938] Loss_D: 0.1002 Loss_G: 8.6717 D(x): 0.9794 D(G(z)): 0.0650 / 0.0003
[0/25][12/938]

# MosaiQ Example

In [13]:
def get_train_data(dataset='MNIST', ds_class=5):
    if dataset == 'MNIST':
        train_loader = torch.utils.data.DataLoader(datasets.MNIST('../mnist', 
                                                                download=True, 
                                                                train=True,
                                                                
                                                                transform=transforms.Compose([
                                                                    torchvision.transforms.ToTensor(),
                                                                    transforms.Lambda(torch.flatten),
                                                                ])), 
                                                batch_size=10000, 
                                                shuffle=True)
    elif dataset == 'Fashion':
        train_loader = torch.utils.data.DataLoader(datasets.FashionMNIST('../fashion', 
                                                                download=True, 
                                                                train=True,
                                                                
                                                                transform=transforms.Compose([
                                                                    torchvision.transforms.ToTensor(),
                                                                    transforms.Lambda(torch.flatten),
                                                                ])), 
                                                batch_size=10000, 
                                                shuffle=True)
    train_data = []
    label_to_keep = ds_class
    label_to_keep_name = str(label_to_keep)
    for (data, labels) in train_loader:
        for x, y in zip(data, labels):
            if y == label_to_keep:
                train_data.append(x.numpy())

    return label_to_keep_name, train_data

In [14]:
def scale_data(data, scale=None, dtype=np.float32):
    if scale is None:
        scale = [-1, 1]
    min_data, max_data = [float(np.min(data)), float(np.max(data))]
    min_scale, max_scale = [float(scale[0]), float(scale[1])]
    data = ((max_scale - min_scale) * (data - min_data) / (max_data - min_data)) + min_scale
    return data.astype(dtype)

In [15]:
# Function from https://machinelearningmastery.com/how-to-implement-the-frechet-inception-distance-fid-from-scratch/
def calculate_fid(act1, act2):
    mu1, sigma1 = act1.mean(axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = act2.mean(axis=0), np.cov(act2, rowvar=False)
    ssdiff = np.sum((mu1 - mu2)**2.0)
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return fid

In [19]:
label_to_keep_name, train_data = get_train_data(dataset='MNIST')
train_data = scale_data(np.array(train_data), [0,1])
image_size = 5  
batch_size = 8
pca_dims=40
n_qubits = 5  
q_depth = 6 
n_generators = 8

In [17]:
pca = PCA(n_components=pca_dims)
pca_data_full = pca.fit_transform(train_data)
ordering = []

In [18]:
pca_data_full.shape, train_data.shape

((5421, 40), (5421, 784))

In [25]:
for i in range(8):
    k = 4*i
    l = [i, 39-k, 38-k, 37-k, 36-k]
    ordering.append(l)
pca_min, pca_max = np.min(pca_data_full), np.max(pca_data_full)

In [26]:
full_train_data = [(i,j) for i,j in zip(scale_data(pca_data_full), train_data)]

transform = transforms.Compose([transforms.ToTensor()])
dataloader = torch.utils.data.DataLoader(
    scale_data(pca_data_full), batch_size=batch_size, shuffle=True, drop_last=True
)

In [27]:
class Discriminator_MosaiQ(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(pca_dims, 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)

In [28]:
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
# @qml.qnode(dev, interface="jax", diff_method="parameter-shift")
def quantum_circuit_MosaiQ(inputs, weights):
    # weights = weights.reshape(q_depth, n_qubits)
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
        qml.RX(inputs[i], wires=i)
    # qml.AngleEmbedding(features=inputs, wires=range(n_qubits), rotation='Y')
    # qml.AngleEmbedding(features=inputs, wires=range(n_qubits), rotation='X')
    
    for i in range(q_depth):
        for y in range(n_qubits):
            qml.RY(weights[i][y], wires=y)
        for y in range(n_qubits - 1):
            qml.CZ(wires=[y, y + 1])
    return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]

In [51]:
n = n_qubits
nlayers = q_depth

def tc_MosaiQ(inputs, weights, index_rot):
    c = tc.Circuit(n)
    
    for i in range(n):
        c.ry(i, theta=inputs[i])
        c.rx(i, theta=inputs[i])

    # for i in range(n):
    #     c.ry(i, theta=index_rot*2*np.pi)
    #     c.rx(i, theta=index_rot*2*np.pi)
    
    for j in range(nlayers):
        for i in range(n):
            c.ry(i, theta=weights[j][i])
        for i in range(n-1):
            c.cz(i, i+1)

    # res = []
    # for i in range(n):
    #     ypred = c.expectation([tc.gates.x(), [i]]) 
    #     res.append(K.real(ypred))
    
    # return torch.as_tensor(K.stack(res))

    res = []
    for i in range(n):
    # if True:
        ypred = c.expectation_ps(x=[i])
        ypred = K.real(ypred)
        res.append(ypred)
    
    # return torch.tensor(res)
    return jnp.array(res)
    # return torch.tensor(ypred)


# Wrap the function into pytorch form but with tensorflow speed!
qpred_torch = tc.interfaces.torch_interface(tc_MosaiQ, jit=True)

In [52]:
class QuantumGenerator_MosaiQ(nn.Module):
    def __init__(self, n_generators, q_delta=1, env='simulation'):
        super().__init__()

        self.q_params = nn.ParameterList(
            [
                nn.Parameter(q_delta * torch.rand(q_depth, n_qubits), requires_grad=True)
                for _ in range(1)
            ]
        )

        self.n_generators = n_generators
        self.env = env

        # weight_shapes = {"weights": (q_depth, n_qubits)}
        # if env == "real":
        #     self.qcircuit = qml.qnn.TorchLayer(quantum_cirtui_real_machine, weight_shapes).to(device)
        # else:
        #     self.qcircuit = qml.qnn.TorchLayer(quantum_circuit_MosaiQ, weight_shapes).to(device)

    def forward(self, x):
        images = []
        patch_size = image_size
        images = torch.Tensor(x.size(0), 0).to(device)

        # x_jax = jnp.array(x.detach().cpu().numpy())
        # all_patches = []

        # print("shape of x:", x.shape, patch_size)
        
        for params in self.q_params:
            patches = torch.Tensor(0, patch_size).to(device)
            for elem_idx, elem in enumerate(x):
                # # when using PennyLane
                # f = quantum_circuit_MosaiQ(elem, params)
                # if self.env == 'Real':
                #     f = quantum_cirtui_real_machine(elem, params)
                # f = torch.tensor(f)

                # # when using TensorCircuit-ng
                f = qpred_torch(elem, params, torch.tensor(elem_idx/8))
                
                q_out = f.float().unsqueeze(0).to(device)
                patches = torch.cat((patches, q_out))
            flattened_order =  [j for sub in ordering for j in sub]
            patches = torch.flatten(patches)
            patches = patches[flattened_order] # Rearrange order of pca components
            patches = patches.reshape(batch_size, patch_size)
            images = torch.cat((images, patches), 1)
        return images

In [53]:
lrG = 0.3
lrD = 0.05
num_iter = 1

gen_losses = []
disc_losses = []
discriminator = Discriminator_MosaiQ().to(device)
generator = QuantumGenerator_MosaiQ(n_generators).to(device)
criterion = nn.BCELoss()
optD = optim.SGD(discriminator.parameters(), lr=lrD)
optG = optim.SGD(generator.parameters(), lr=lrG)
real_labels = torch.full((batch_size,), 1.0, dtype=torch.float, device=device)
fake_labels = torch.full((batch_size,), 0.0, dtype=torch.float, device=device)
counter = 0

noise_upper_bound = math.pi/8

In [54]:
def relu(x):
    return x * (x > 0)
def get_noise_upper_bound(gen_loss, disc_loss, original_ratio):
    R = disc_loss.detach().cpu().numpy()/gen_loss.detach().cpu().numpy()
    return math.pi/8 + (5 *math.pi / 8) * relu(np.tanh((R - (original_ratio))))

In [55]:
original_ratio = None
upper_bounds = [math.pi/8]
results = []
generated_images = []

## train

In [56]:
print('Training...')
for e in tqdm(range(num_iter)):
    for i, train_pair in enumerate(dataloader):
        pca_data = train_pair
        data = pca_data.reshape(batch_size, pca_dims)
        real_data = data.to(device).to(torch.float32)
        noise = torch.rand(batch_size, n_qubits, device=device) * noise_upper_bound

        st = time.time()
        fake_data = generator(noise)
        print(i, "time taken:", time.time() - st)
        
        discriminator.zero_grad()
        outD_real = discriminator(real_data).view(-1)
        outD_fake = discriminator(fake_data.detach()).view(-1)
        errD_real = criterion(outD_real, real_labels)
        errD_fake = criterion(outD_fake, fake_labels)
        errD_real.backward()
        errD_fake.backward()
        errG = criterion(outD_fake, real_labels)
        errD = errD_real + errD_fake
        
        # gen_losses.append(errG.detach().cpu().numpy())
        # disc_losses.append(errD.detach().cpu().numpy())
        
        gen_losses.append(errG.detach().cpu().numpy())
        disc_losses.append(errD.detach().cpu().numpy())

        optD.step()

        # Train the generator
        generator.zero_grad()
        outD_fake = discriminator(fake_data).view(-1)
        errG = criterion(outD_fake, real_labels)
        errG.backward()
        optG.step()
        if original_ratio is None:
            original_ratio = errD.detach().cpu().numpy()/errG.detach().cpu().numpy()
        noise_upper_bound = get_noise_upper_bound(errG, errD, original_ratio)
        upper_bounds.append(noise_upper_bound)
        np.save(f'upper_bounds_{label_to_keep_name}', upper_bounds)
        counter += 1      
        if counter % 20 == 0:  
            test_images = generator(noise).detach().cpu().numpy()
            test_images = pca.inverse_transform(test_images)
            fid = calculate_fid(test_images.reshape([batch_size, 784]), train_data)
            test_images = scale_data(test_images,[0,1])
            real_images = []
            np.save(f'gen_loss_{label_to_keep_name}', gen_losses)
            np.save(f'disc_loss_{label_to_keep_name}', disc_losses)
            from PIL import Image 
            im = np.reshape(test_images[0], [28, 28])
            new_im = np.zeros([28,28])
            for i in range(28):
                for j in range(28):
                    if im[i][j] > .5:
                        new_im[i][j] = 0.0
                    else:
                        new_im[i][j] = 1.0
            im = Image.fromarray(np.uint8(255-(new_im*255)))
            
            # im = im.save(os.path.join("../QGAN/gen_images_dist",f"{label_to_keep_name}_{counter}.png"))  # original
            # im = im.save(os.path.join("../QGAN/gen_images_dist simple rot",f"{label_to_keep_name}_{counter}.png"))  # naive rotation
            im = im.save(os.path.join("../QGAN/gen_images_dist same generator no noise encoder",f"{label_to_keep_name}_{counter}.png"))  # naive rotation
            
            torch.save(generator.state_dict(), f"generator_{label_to_keep_name}")
            torch.save(discriminator.state_dict(), f"disc_{label_to_keep_name}")

Training...


  0%|                                                                       | 0/1 [00:00<?, ?it/s]

0 time taken: 4.887498378753662


  0%|                                                                       | 0/1 [00:05<?, ?it/s]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (8x5 and 40x64)